# Advanced Artificial Intelligence Task 1:

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import sys
sys.path.append("..")
sys.path.append(".")
from experiment_configs.task_1_config import(
    ncf_baseline,
    lstm_baseline,
    sasrec_baseline
)

from models.ncf import NCF
from models.lstm import LSTM
from models.sasrec import SASRec

In [14]:
#EXP=ncf_baseline
EXP=lstm_baseline
#EXP=sasrec_baseline

In [15]:
df=pd.read_csv("data/insta_clean_data.csv")
df=df.sample(n=200000,random_state=42)
df=df.sort_values(["user_id","order_id","cart_position"])

df=df.dropna(subset=["user_id","product_name"])

In [16]:
user_encoder=LabelEncoder()
item_encoder=LabelEncoder()

df["user_id"]=user_encoder.fit_transform(df["user_id"])
df["item_id"]=item_encoder.fit_transform(df["product_name"])

num_users=df["user_id"].nunique()
num_items=df["item_id"].nunique()

print("Users:",num_users,"Items:",num_items)

Users: 90182 Items: 22355


In [17]:
user_sequences=df.groupby("user_id")["item_id"].apply(list)

user_sequences=[seq for seq in user_sequences if len(seq)>2]

In [18]:
train_sequences=[]
test_sequences=[]

for seq in user_sequences:
    split=int(len(seq)*0.8)
    train_sequences.append(seq[:split])
    test_sequences.append(seq[split:])

In [19]:
class NCFDataset(Dataset):
    def __init__(self,df):
        self.users=df["user_id"].values
        self.items=df["item_id"].values
        self.labels=df["is_reorder"].values

    def __len__(self):
        return len(self.users)
    
    def __getitem__(self,idx):
        return(
            torch.tensor(self.users[idx],dtype=torch.long),
            torch.tensor(self.items[idx],dtype=torch.long),
            torch.tensor(self.labels[idx],dtype=torch.float)
        )

class SequenceDataset(Dataset):
    def __init__(self,sequences,max_len=10):
        self.data=[]
        self.max_len=max_len

        for seq in sequences:
            for i in range(1,len(seq)):
                input_seq=seq[max(0,i-max_len):i]
                target=seq[i]
                self.data.append((input_seq,target))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        seq,target=self.data[idx]
        seq=[0]*(self.max_len-len(seq))+seq
        return(
            torch.tensor(seq,dtype=torch.long),
            torch.tensor(target,dtype=torch.long)
        )

In [20]:
if EXP.architecture=="ncf":
    dataset=NCFDataset(df)
elif EXP.architecture in ["lstm","sasrec"]:
    dataset=SequenceDataset(train_sequences,max_len=EXP.model.max_seq_len)

loader=DataLoader(dataset,batch_size=EXP.training.batch_size,shuffle=True)

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if EXP.architecture=="ncf":
    model=NCF(num_users,num_items,EXP.model.embedding_dim)
elif EXP.architecture=="lstm":
    model=LSTM(
        num_items,
        embedding_dim=EXP.model.embedding_dim,
        hidden_dim=EXP.model.hidden_dim
    )
elif EXP.architecture=="sasrec":
    model=SASRec(
        num_items,
        embedding_dim=EXP.model.embedding_dim,
        num_heads=EXP.model.num_heads,
        num_layers=EXP.model.num_layers
    )
model=model.to(device)

In [22]:
optimizer=torch.optim.Adam(model.parameters(),lr=EXP.training.learning_rate)

if EXP.architecture=="ncf":
    loss_fn=torch.nn.BCEWithLogitsLoss()
else:
    loss_fn=torch.nn.CrossEntropyLoss()

for epoch in range(EXP.training.max_epochs):
    total_loss=0
    for batch in loader:
        optimizer.zero_grad()

        if EXP.architecture == "ncf":
            user,item,label=batch
            user,item,label=user.to(device),item.to(device),label.to(device)

            pred=model(user,item)
            loss=loss_fn(pred,label)

        else:
            seq,target=batch
            seq,target=seq.to(device),target.to(device)
            
            pred=model(seq)
            loss=loss_fn(pred,target)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch}: Loss {total_loss/len(loader):.4f}")

Epoch 0: Loss 8.9139
Epoch 1: Loss 8.3190
Epoch 2: Loss 7.9906
Epoch 3: Loss 7.5601
Epoch 4: Loss 7.1421
Epoch 5: Loss 6.7606
Epoch 6: Loss 6.4095
Epoch 7: Loss 6.0860
Epoch 8: Loss 5.7857
Epoch 9: Loss 5.5109


In [24]:
def recall_at_k(model,sequences,k=5):
    model.eval()
    correct=0
    total=0
    for seq in sequences:
        if len(seq)<2:
            continue
        
        input_seq=seq[:-1]
        true_item=seq[-1]
        input_seq=input_seq[-10:]
        input_seq=[0]*(10-len(input_seq))+input_seq
        input_tensor=torch.tensor(input_seq).unsqueeze(0).to(device)

        scores=model(input_tensor   )
        top_k=scores.topk(k).indices.squeeze().tolist()

        if true_item in top_k:
            correct+=1
        total+=1

    return correct/total

print("Recall@5:", recall_at_k(model,test_sequences,k=5))

Recall@5: 0.013717421124828532
